In [0]:
from pyspark.sql.functions import *

In [0]:
max_dh_bronze = (
    spark.read.table('api_football.bronze.matches_raw')
    .agg(max('dh_processing_br').alias('max_dh_bronze'))
    .collect()[0]['max_dh_bronze'] #transforma de dataframe para variável
) #coletando o máximo registro e transformando em uma váriavel para usar como filtro
df_silver = (
    spark.read.table('api_football.silver.goals')
)
df_matches_raw = (
    spark.read.table('api_football.bronze.matches_raw').alias('mr')
     .join(df_silver.alias('s'), col('s.match_id') == col('mr.match_id'), 'leftanti') # join para pegar os dados que não estão no silver
    .filter(col('mr.dh_processing_br') == max_dh_bronze)
    .filter(col('mr.match_status') == 'Finished')
)

# display(df_matches_raw.count())

In [0]:
#df_matches_raw.printSchema()

In [0]:
df_goals_exploded = (
    df_matches_raw
    .select(
        "match_id",
        "match_hometeam_id",
        "match_hometeam_name",
        "match_awayteam_id",
        "match_awayteam_name",
        "dh_processing_br",
        explode("goalscorer").alias("goal")
    )
)

#display(df_goals_exploded)


In [0]:
df_goals = (
    df_goals_exploded
    .select(
        col("match_id"),
        when(
            col("goal.home_scorer") != "",
            col("goal.home_scorer")
        ).otherwise(
            col("goal.away_scorer")
        ).alias("player_name"),
        when(
            col("goal.home_scorer") != "",
            col("goal.home_assist")
        ).otherwise(
            col("goal.away_assist")
        ).alias("assist_name"),
        col("goal.time").alias("score_time"),
         col("goal.score").alias("score"),
            when(
            col("goal.home_scorer") != "",
            col("match_hometeam_id")
        ).otherwise(
            col("match_awayteam_id")
        ).alias("team_id"),
            when(
            col("goal.home_scorer") != "",
            col("match_hometeam_name")
        ).otherwise(
            col("match_awayteam_name")
        ).alias("team_name"),
            when(
            col("goal.home_scorer") != "",
            lit(True)
        ).otherwise(
            lit(False)
        ).alias("is_home_team"),
         col('dh_processing_br').alias('dh_processing_bronze_br'),
        expr('current_timestamp()-interval 3 hours').alias('dh_processing_br')
    )
)

#display(df_goals)

In [0]:
df_goals.write.mode('append').saveAsTable('api_football.silver.goals')

In [0]:
%sql
--select count(*) from api_football.silver.goals